# v1: Exploratory data analysis and depot-shift diagnosis

**Version:** v1  
**Kaggle submission:** none (analysis; its fixes are used by every later version)

**Questions:** what does the target look like, is the label reliable (censoring), how are missing values related to failure, and how do the train depot and the test depot differ? The fixes found here (histogram realignment, dropped columns) are in `load()` in every notebook.

*Self-contained: needs only `../data/train.csv` and `../data/test.csv`. Part of the Fleet Maintenance project (Kaggle `mlprojectfleet2`); the cost, decision rule and validation protocol are derived in `fleet_maintenance.ipynb`.*

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, cross_val_predict

DATA, OUT = Path("../data"), Path("outputs"); OUT.mkdir(exist_ok=True)
SEED = 42
INSPECT, UNDER, OVER = 10.0, 0.20, 0.02          # inspection cost; shortage cost / unit; unused-budget cost / unit
Q_STAR = UNDER / (UNDER + OVER)                  # newsvendor critical ratio 0.909
HISTB = [f"s{i:03d}" for i in range(54, 64)]     # 10-bin histogram that is shifted by one bin in test

def load(raw=False):
    """train/test with the v1 fixes: realign s054-s063 (test bin k+1 == train bin k), drop s047 (all NaN in
    test) and s146 (constant). raw=True returns the files untouched."""
    tr, te = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
    if not raw:
        te[HISTB[:-1]] = te[HISTB[1:]].to_numpy(); te[HISTB[-1]] = np.nan
        tr, te = tr.drop(columns=["s047", "s146"]), te.drop(columns=["s047", "s146"])
    return tr, te, [c for c in tr.columns if c.startswith("s")]

tr, te, S = load()
dev_idx, ho_idx = train_test_split(np.arange(len(tr)), test_size=0.2, stratify=tr.failure, random_state=SEED)
dev, ho = tr.iloc[dev_idx].reset_index(drop=True), tr.iloc[ho_idx].reset_index(drop=True)   # ho: locked 20% hold-out
y, c = dev.failure.to_numpy(), dev.repair_cost.fillna(0).to_numpy() * dev.failure.to_numpy()
y_ho, c_ho = ho.failure.to_numpy(), ho.repair_cost.fillna(0).to_numpy() * ho.failure.to_numpy()
c_all = tr.repair_cost.fillna(0).to_numpy() * tr.failure.to_numpy()
F = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # the 5 fixed folds
print(f"train {tr.shape}  test {te.shape}  sensors {len(S)}  dev {len(dev)} / hold-out {len(ho)}  failures {int(y.sum())}")

train (56000, 172)  test (20000, 169)  sensors 168  dev 44800 / hold-out 11200  failures 318


In [2]:
def vehicle_cost(c, flag, b):
    return np.where(flag == 1, INSPECT + UNDER * np.maximum(c - b, 0) + OVER * np.maximum(b - c, 0), c)

def per_vehicle(c, flag, b):
    return float(vehicle_cost(np.asarray(c, float), np.asarray(flag), np.asarray(b, float)).mean())

fold_cost = lambda flag, b: np.array([per_vehicle(c[v], flag[v], b[v]) for _, v in F])

def write_submission(ids, flag, budget, name):
    flag, budget = np.asarray(flag).astype(int), np.maximum(np.asarray(budget, float), 0)
    assert len(ids) == 20000 and ids.is_unique and set(flag) <= {0, 1} and np.isfinite(budget).all()
    pd.DataFrame({"id": ids, "flag": flag, "budget": budget.round(2)}).to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"wrote outputs/submission_{name}.csv: flagged {flag.mean():.2%} of test vehicles")

def decide_global(p, fail_costs):
    """Same repair-cost distribution F (empirical failure costs) for every vehicle."""
    p = np.clip(np.asarray(p, float), 0, 1); fc = np.sort(np.asarray(fail_costs, float)); mu = fc.mean()
    b = np.zeros_like(p); need = p > 1 - Q_STAR
    b[need] = np.quantile(fc, 1 - (1 - Q_STAR) / p[need])                       # newsvendor budget
    fail = (UNDER * np.maximum(fc[None, :] - b[:, None], 0) + OVER * np.maximum(b[:, None] - fc[None, :], 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b   # flag iff cheaper than not flagging

def decide_pv(p, m, resid, fail_costs=None, K=99, scale=None):
    """Per-vehicle cost distribution: log c | fail ~ m_i + scale_i * residuals (out-of-fold residuals).
    fail_costs given -> budget from that GLOBAL distribution (ablation); scale -> per-vehicle spread (v9)."""
    p, m = np.clip(np.asarray(p, float), 0, 1), np.asarray(m, float)
    sc = np.ones_like(m) if scale is None else np.asarray(scale, float)
    M = np.exp(m[:, None] + sc[:, None] * np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :]); mu = M.mean(1)
    b = np.zeros_like(p); need = p > 1 - Q_STAR; level = 1 - (1 - Q_STAR) / p[need]
    b[need] = (np.exp(m[need] + sc[need] * np.quantile(resid, level)) if fail_costs is None
               else np.quantile(np.asarray(fail_costs, float), level))
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b

class Platt:
    def fit(self, p, y):
        self.m = LogisticRegression(C=1e6, max_iter=1000).fit(self._x(p), y); return self
    def predict(self, p): return self.m.predict_proba(self._x(p))[:, 1]
    @staticmethod
    def _x(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y); return self
    def predict(self, p): return self.m.predict(p)

make_cal = lambda kind: {"platt": Platt, "iso": Iso}[kind]()

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, n_jobs=-1, random_state=SEED)                         # failure classifier (v2..)
CM = dict(n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=5, n_jobs=-1, random_state=SEED)                             # repair-cost model (v5)
inner = KFold(5, shuffle=True, random_state=SEED)

def cost_model(params, Xa, lc, Xb):
    """XGBoost on log repair cost (failures only). Returns predictions for Xb and OUT-OF-FOLD residuals of Xa."""
    resid = lc - cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner)
    return xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb), resid

def fit_oof(X, params=PARAMS):
    """5-fold out-of-fold failure probabilities on dev, and probabilities for test from a model fit on all train."""
    o = np.zeros(len(dev))
    for a, b in F:
        o[b] = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    return o

shift = lambda p, k: k * p / (1 - p) / (1 + k * p / (1 - p))       # prior shift by odds factor k

In [3]:
tr0, te0, S0 = load(raw=True); f = tr0[tr0.failure == 1]
print(f"failures: {len(f)} / {len(tr0)} = {tr0.failure.mean():.3%}")
print("repair cost quantiles 5/25/50/75/90/95/99:", f.repair_cost.quantile([.05, .25, .5, .75, .9, .95, .99]).round(0).tolist(), " max", f.repair_cost.max())
top = f.repair_cost.sort_values(ascending=False)
print(f"top 10% of failures carry {top.iloc[:len(top)//10].sum() / top.sum():.1%} of all repair cost")
print(f"flag threshold with mean cost mu={f.repair_cost.mean():.0f}: p*mu > 12.5  ->  p > {12.5 / f.repair_cost.mean():.2%}")

failures: 397 / 56000 = 0.709%
repair cost quantiles 5/25/50/75/90/95/99: [5.0, 71.0, 176.0, 488.0, 1094.0, 1693.0, 6847.0]  max 8000.0
top 10% of failures carry 53.4% of all repair cost
flag threshold with mean cost mu=492: p*mu > 12.5  ->  p > 2.54%


### Censoring: `followup_days`

In [4]:
bins = pd.cut(tr0.followup_days, [0, 90, 180, 270, 364, 365])
print(tr0.groupby(bins, observed=True).failure.agg(["mean", "size"]).round(4).to_string())
print(f"\nhealthy vehicles with < 365 days of follow-up: {((tr0.failure == 0) & (tr0.followup_days < 365)).sum() / (tr0.failure == 0).sum():.1%}")

                 mean   size
followup_days               
(0, 90]        0.0007   2705
(90, 180]      0.0048   3144
(180, 270]     0.0085   3050
(270, 364]     0.0191   3295
(364, 365]     0.0066  43806

healthy vehicles with < 365 days of follow-up: 21.7%


### Missing values

In [5]:
nan_tr, nan_te = tr0[S0].isna().mean(axis=1), te0[S0].isna().mean(axis=1)
print(f"mean missing fraction: train {nan_tr.mean():.3f}  test {nan_te.mean():.3f}")
q = pd.qcut(nan_tr.rank(method="first"), 4, labels=["Q1 (least missing)", "Q2", "Q3", "Q4 (most missing)"])
print("failure rate by missingness quartile:", tr0.groupby(q, observed=True).failure.mean().round(4).to_dict())
print("constant columns in train:", [col for col in S0 if tr0[col].nunique(dropna=True) <= 1])
print("columns 100% missing in test:", [col for col in S0 if te0[col].isna().all()])

mean missing fraction: train 0.083  test 0.091
failure rate by missingness quartile: {'Q1 (least missing)': 0.0061, 'Q2': 0.0002, 'Q3': 0.0014, 'Q4 (most missing)': 0.0206}


constant columns in train: ['s146']
columns 100% missing in test: ['s047']


### Depot shift: adversarial validation and the shifted histogram

In [6]:
def adversarial(a, b, cols, tag):
    X = pd.concat([a[cols], b[cols]], ignore_index=True); z = np.r_[np.zeros(len(a)), np.ones(len(b))]
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=SEED)
    p = cross_val_predict(clf, X, z, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    imp = pd.Series(clf.fit(X, z).feature_importances_, index=cols).sort_values(ascending=False)
    print(f"[{tag}] adversarial AUC = {roc_auc_score(z, p):.4f}; most important:", imp.head(6).round(3).to_dict()); return p

adversarial(tr0, te0, S0, "raw files")
print("\nKS(train bin k, test bin k)   :", [round(float(ks_2samp(tr0[HISTB[i]].dropna(), te0[HISTB[i]].dropna()).statistic), 3) for i in range(9)])
print("KS(train bin k, test bin k+1) :", [round(float(ks_2samp(tr0[HISTB[i]].dropna(), te0[HISTB[i + 1]].dropna()).statistic), 3) for i in range(9)], " <- test histogram is shifted by one bin")

[raw files] adversarial AUC = 1.0000; most important: {'s062': 0.41999998688697815, 's047': 0.29899999499320984, 's012': 0.08500000089406967, 's054': 0.0689999982714653, 's056': 0.02500000037252903, 's156': 0.020999999716877937}

KS(train bin k, test bin k)   : [0.982, 0.719, 0.56, 0.277, 0.124, 0.459, 0.414, 0.607, 0.804]


KS(train bin k, test bin k+1) : [0.026, 0.041, 0.03, 0.067, 0.032, 0.026, 0.02, 0.015, 0.039]  <- test histogram is shifted by one bin


In [7]:
cols = [col for col in S if col != HISTB[-1]]            # tr/te: after the fixes
ks = pd.Series({col: ks_2samp(tr[col].dropna(), te[col].dropna()).statistic for col in cols if tr[col].notna().sum() > 100 and te[col].notna().sum() > 100})
print("largest remaining KS after fixes:", ks.sort_values(ascending=False).head(8).round(3).to_dict())
adversarial(tr, te, cols, "after fixes")
adversarial(tr, te, [col for col in cols if col not in ("s120", "s122", "s126", "s128")], "after fixes, minus s120/s122/s126/s128")

largest remaining KS after fixes: {'s120': 0.74, 's122': 0.312, 's166': 0.073, 's057': 0.067, 's108': 0.067, 's121': 0.058, 's100': 0.05, 's055': 0.041}


[after fixes] adversarial AUC = 0.9999; most important: {'s128': 0.2709999978542328, 's126': 0.2639999985694885, 's120': 0.2070000022649765, 's012': 0.03500000014901161, 's057': 0.01600000075995922, 's097': 0.014000000432133675}


[after fixes, minus s120/s122/s126/s128] adversarial AUC = 0.9371; most important: {'s057': 0.1080000028014183, 's055': 0.08799999952316284, 's062': 0.07599999755620956, 's038': 0.03500000014901161, 's154': 0.032999999821186066, 's149': 0.019999999552965164}


array([0.15119466, 0.02556839, 0.00308246, ..., 0.29502255, 0.9687223 ,
       0.99672556], dtype=float32)

## Findings and caveats

- **Target:** 397 failures out of 56,000 (**0.71%**). Repair cost is heavy-tailed (median 176, mean 492, p99 6,847, max 8,000); the top 10% of failures carry 53% of the cost. With μ = 492 the flag rule is **p > ~2.5%**, about 3.6× the base rate, far below 0.5.
- **Censoring:** 22% of healthy vehicles were followed for < 365 days. The failure rate rises with follow-up (0.07% for ≤ 90 days → 1.9% for 271-364 days), so records of failing vehicles appear to stop near the failure: `followup_days` is partly determined *by* the outcome. It is **never used as a feature** (and it is not in test).
- **Missingness predicts failure:** the most-missing quartile fails at 2.06% vs 0.02-0.6% for the others. NaN stays NaN (XGBoost handles it natively).
- **Depot shift:** adversarial AUC on the raw files is **1.000**. Two causes are data errors and are fixed: `s047` is 100% missing in test (dropped; `s146` is constant, dropped), and the 10-bin histogram `s054-s063` is **shifted by one bin in test** (test bin k+1 matches train bin k, KS ≤ 0.07; realigned). After the fixes the AUC is still **0.9999**: `s120` and `s122` are level-shifted (a counter that runs higher, ~×2-2.7) and `s126`/`s128` differ in a way a rounding hypothesis did not explain; without the four, AUC is 0.937 (a diffuse shift remains).

**Caveats**
- The realignment is inferred from distributions, not from documentation. It is the simplest explanation (a shift of exactly one bin matches all 9 pairs; two does not), but it cannot be confirmed.
- Adversarial AUC shows the depots *differ*, not that the differences *matter for failure*. Whether to drop the drifted sensors is decided on cost (v3b), not on AUC.
- With 397 positives every per-group failure rate has wide confidence intervals (the ≤ 90-day group has only 2 failures).